In [0]:
# UrbanFlow - Phase 1.1
# Step 1: Verify source files in Unity Catalog Volume

base_path = "/Volumes/urbanflow/raw_data/source_files"

print("=== YELLOW TAXI FILES ===")
display(dbutils.fs.ls(f"{base_path}/yellow_taxi/"))

print("=== REFERENCE FILES ===")
display(dbutils.fs.ls(f"{base_path}/reference/"))

In [0]:
# Read January 2025 Yellow Taxi data

file_path = f"{base_path}/yellow_taxi/yellow_tripdata_2025-01.parquet"

df_january = spark.read.parquet(file_path)

print("=== DATASET SCHEMA ===")
df_january.printSchema()

print("=== NUMBER OF COLUMNS ===")
print(len(df_january.columns))

print("=== FIRST 5 ROWS ===")
display(df_january.limit(5))

In [0]:

# UrbanFlow - Phase 1.1.3
# Compare schemas across January, February and March 2025

months = ["2025-01", "2025-02", "2025-03"]

schemas = {}

for month in months:
    path = f"{base_path}/yellow_taxi/yellow_tripdata_{month}.parquet"
    
    df = spark.read.parquet(path)
    
    schemas[month] = {
        field.name: field.dataType.simpleString()
        for field in df.schema.fields
    }

    print(f"{month}: {len(df.columns)} columns")

print("\n=== SCHEMA COMPARISON ===")

reference = schemas["2025-01"]

for month in months[1:]:
    current = schemas[month]

    if current == reference:
        print(f"{month}: SAME SCHEMA AS JANUARY")
    else:
        print(f"{month}: SCHEMA DIFFERENCES DETECTED")

        for col in sorted(reference.keys() | current.keys()):
            if reference.get(col) != current.get(col):
                print(
                    f"  {col}: "
                    f"January={reference.get(col)} | "
                    f"{month}={current.get(col)}"
                )


In [0]:

# UrbanFlow - Phase 1.1.4
# Read Taxi Zone Lookup CSV

zones_path = f"{base_path}/reference/taxi_zone_lookup.csv"

df_zones = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(zones_path)
)

print("=== TAXI ZONES SCHEMA ===")
df_zones.printSchema()

print("\n=== NUMBER OF COLUMNS ===")
print(len(df_zones.columns))

print("\n=== FIRST 5 ROWS ===")
display(df_zones.limit(10))
